# Лабораторная работа №5: Тестирование качества работы моделей машинного обучения

**Цель:** Применить `pytest` для автоматического тестирования качества линейной регрессии на качественных и зашумленных данных.

### Установка зависимостей
Эта ячейка установит все необходимые библиотеки для выполнения работы прямо в текущее окружение ноутбука.

In [ ]:
%pip install pandas scikit-learn matplotlib pytest joblib

### Генерация данных и обучение модели
Создаем три датасета, визуализируем данные и обучаем модель.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import joblib

os.makedirs('lab5', exist_ok=True)

np.random.seed(42)

def generate_good_data(n_samples=100):
    """Генерация качественных данных для линейной регрессии"""
    xs = np.linspace(0, 10, n_samples)
    ys = xs + np.random.random(n_samples) * 2 - 1
    return pd.DataFrame({'x': xs, 'y': ys})

def generate_noisy_data(n_samples=100):
    """Генерация данных с шумом (выбросами)"""
    xs = np.linspace(0, 10, n_samples)
    ys = xs + np.random.random(n_samples) * 2 - 1
    ys[25:45] *= 2
    return pd.DataFrame({'x': xs, 'y': ys})

df_good1 = generate_good_data()
df_good2 = generate_good_data()
df_good3 = generate_good_data()

df_good1.to_csv('lab5/data_1.csv', index=False)
df_good2.to_csv('lab5/data_2.csv', index=False)
df_good3.to_csv('lab5/data_3.csv', index=False)

df_noisy = generate_noisy_data()
df_noisy.to_csv('lab5/noisy_data.csv', index=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(df_good1['x'], df_good1['y'])
ax1.set_title("Качественные данные (df_good1)")
ax2.scatter(df_noisy['x'], df_noisy['y'], color='orange')
ax2.set_title("Данные с шумом (df_noisy)")
plt.show()

X_train = df_good1[['x']]
y_train = df_good1['y']

model = LinearRegression()
model.fit(X_train, y_train)

joblib.dump(model, 'lab5/linear_model.pkl')
print("Модель обучена и сохранена в lab5/linear_model.pkl")

### Тестирование

Проверяем загруженную модель на двух хороших датасетах и на одном зашумленном.

In [ ]:
%%writefile lab5/test_model.py
import pytest
import pandas as pd
import joblib
from sklearn.metrics import mean_squared_error, r2_score

@pytest.fixture(scope="module")
def model():
    return joblib.load('lab5/linear_model.pkl')

def evaluate_model(model, data_path):
    df = pd.read_csv(data_path)
    X = df[['x']]
    y_true = df['y']
    y_pred = model.predict(X)

    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return mse, r2

MSE_THRESHOLD = 1.5
R2_THRESHOLD = 0.85

def test_model_on_data_2(model):
    mse, r2 = evaluate_model(model, 'lab5/data_2.csv')
    assert mse < MSE_THRESHOLD, f"Слишком высокая ошибка MSE: {mse:.2f}"
    assert r2 > R2_THRESHOLD, f"Низкий R2 Score: {r2:.2f}"

def test_model_on_data_3(model):
    mse, r2 = evaluate_model(model, 'lab5/data_3.csv')
    assert mse < MSE_THRESHOLD, f"Слишком высокая ошибка MSE: {mse:.2f}"
    assert r2 > R2_THRESHOLD, f"Низкий R2 Score: {r2:.2f}"

def test_model_on_noisy_data(model):
    mse, r2 = evaluate_model(model, 'lab5/noisy_data.csv')
    assert mse < MSE_THRESHOLD, f"Обнаружен шум, MSE превысил порог: {mse:.2f} > {MSE_THRESHOLD}"
    assert r2 > R2_THRESHOLD, f"Обнару жен шум, R2 слишком низкий: {r2:.2f} < {R2_THRESHOLD}"

### Запуск тестов

In [ ]:
!python -m pytest lab5/test_model.py -v